In [ ]:
modelpath_base_str = "converted_models/"
dataset_name = "mixed"
y_max_float = 0.180
MAX_LENGTH = 512 # 128 or 512

### Choose a model
model_type_str = "multil-i8f32"
model_org_str = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

# model_type_str = "sonoisa-i8f32"
# model_org_str = "sonoisa/sentence-bert-base-ja-mean-tokens-v2"

# model_type_str = "tohoku-bb2-i8f32"
# model_org_str = "cl-tohoku/bert-base-japanese-v2"

# model_type_str = "tohoku-bbwwm-i8f32"
# model_org_str = "cl-tohoku/bert-base-japanese-whole-word-masking"

# model_type_str = "izumil-i8f32"
# model_org_str = "izumi-lab/bert-small-japanese"


modelpath_str = modelpath_base_str + model_type_str


### Choose a dataset
path_dataset_str = '../datasets/jppost/mixed/data_sbert_speed_900.csv'
# path_dataset_str = '../datasets/jppost/mixed/data_sbert_speed_700.csv'
# path_dataset_str = '../datasets/jppost/mixed/data_sbert_speed_200.csv'

print(f"{model_type_str = }")
print(f"{model_org_str = }")
print(f"{path_dataset_str = }")
print(f"{modelpath_str = }")

In [ ]:
import ctranslate2
import numpy as np
import transformers

tokenizer = transformers.AutoTokenizer.from_pretrained(model_org_str)
encoder = ctranslate2.Encoder(modelpath_str, compute_type="int8_float32", device="cpu")

In [ ]:
import time
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression

%matplotlib inline
# plt.style.use('dark_background')
plt.style.use('default')


column_charslen_str = 'the_num_of_chars'
column_tokenslen_str = 'the_num_of_tokens'
column_elapsedtime_str = 'elapsed_time'    


def get_elapsed_times(data_df):
    column_sentence_int = 1
    # column_sentence_str = 'question'
    
    data_df[column_tokenslen_str] = 0
    data_df[column_elapsedtime_str] = 0.
    
    tokens_test = tokenizer(["test encoding"]).input_ids
    encoder.forward_batch(tokens_test)
    
    for row_pdtuple in data_df.itertuples():
        row_index_int = row_pdtuple.Index
        
        sentence_str = row_pdtuple[column_sentence_int]
        tokens_list = tokenizer([sentence_str]).input_ids
        data_df.loc[row_index_int, column_tokenslen_str] = len(tokens_list[0])
        
        start_time = time.time()
        tokens = tokenizer([sentence_str], max_length=MAX_LENGTH, truncation=True).input_ids
        embeddings_list = np.mean(encoder.forward_batch(tokens).last_hidden_state, axis=1)
        end_time = time.time()
        
        elapsed_time = end_time - start_time
        
        if row_index_int % 500 == 0:
            print(row_index_int, elapsed_time)
        
        data_df.loc[row_index_int, column_elapsedtime_str] = elapsed_time
    
    return data_df


def draw_graph(data_df, column_str, color_plot_str='#086972', color_lr_str='#DF5E88'):
    # Set coordinates
    x_max_float = 0.0
    if column_str == column_charslen_str:
        x_max_float = 920.0
    elif column_str == column_tokenslen_str:
        x_max_float = 530.0
    
    x_describe_df = data_df[column_str].describe()
    y_describe_df = data_df[column_elapsedtime_str].describe()
    x_min_float = x_max_float / 36
    x_mid_float = x_max_float / 2
    x_indent_float = x_max_float / 18
    y_incrm_float = y_max_float / 30
    y_title_float = y_max_float - y_incrm_float
    y_stats_float = y_max_float / 3
    y_lr_float = y_max_float / 2
    
    
    # Format data
    x_sr = data_df[column_str].values
    x_df = data_df[[column_str]].values
    y_sr = data_df[column_elapsedtime_str].values
    
    # Linear Regression
    lr = LinearRegression()
    lr.fit(x_df, y_sr)
    
    coef_float = lr.coef_[0]
    intercept_float = lr.intercept_
    
    # Plot
    plt.clf() # Clear the plot
    elapsed_snsplot = sns.jointplot(x=x_sr, y=y_sr, xlim=(0, x_max_float), ylim=(0, y_max_float), color=color_plot_str)
    plt.plot(x_df, lr.predict(x_df), color=color_lr_str)
    
    # Print titles
    plt.text(x_min_float, y_title_float, f'#{dataset_name.upper()}', horizontalalignment='left')
    plt.text(x_min_float, y_title_float - y_incrm_float, f'#{model_type_str}', horizontalalignment='left')
    
    # Print the coefficient and the intercept of Linear Regression
    plt.text(x_mid_float, y_lr_float, '<Linear Regression>', horizontalalignment='left')
    plt.text(x_mid_float + x_indent_float, y_lr_float - y_incrm_float * 1, 'coef', horizontalalignment='left')
    plt.text(x_max_float, y_lr_float - y_incrm_float * 1, '%.8f' % coef_float, horizontalalignment='right')
    plt.text(x_mid_float + x_indent_float, y_lr_float - y_incrm_float * 2, 'intercept', horizontalalignment='left')
    plt.text(x_max_float, y_lr_float - y_incrm_float * 2, '%.8f' % intercept_float, horizontalalignment='right')
    
    # Print stats of x
    plt.text(x_min_float, y_stats_float + y_incrm_float * 0.5, f'<{column_str}>', horizontalalignment='left')
    
    for i, key in enumerate(list(x_describe_df.index)):
        plt.text(x_min_float + x_indent_float, y_stats_float - y_incrm_float * (i + 1), key, horizontalalignment='left')
        plt.text(x_mid_float, y_stats_float - y_incrm_float * (i + 1), '%.8f' % x_describe_df[key], horizontalalignment='right')
        
    # Print stats of y
    plt.text(x_mid_float, y_stats_float + y_incrm_float * 0.5, f'<{column_elapsedtime_str}>', horizontalalignment='left')
    
    for i, key in enumerate(list(x_describe_df.index)):
        plt.text(x_mid_float + x_indent_float, y_stats_float - y_incrm_float * (i + 1), key, horizontalalignment='left')
        plt.text(x_max_float, y_stats_float - y_incrm_float * (i + 1), '%.8f' % y_describe_df[key], horizontalalignment='right')
        

    plt.show()
    
    # Save the plot
    x_type_str = column_str.replace('the_num_of', '')
    filepath_output = 'elapsed_time/elapsed_time_' + model_type_str + x_type_str + '_' + dataset_name + '_h' + str(y_max_float) + '.png'
    # filepath_output = 'elapsed_time/elapsed_time_' + model_type_str + x_type_str + '_' + dataset_name + '900.png'
    elapsed_snsplot.figure.savefig(filepath_output)



In [ ]:
# Load the datasets
data_df = pd.read_csv(path_dataset_str, index_col=0)
# data_df

# Measure elapsed time for encoding each sentence in the datasets
data_df = get_elapsed_times(data_df)
# data_df

# Draw the graphs
draw_graph(data_df, column_charslen_str, color_plot_str='#086972')
draw_graph(data_df, column_tokenslen_str, color_plot_str='#277BC0')
